# 1장 실습 — 차량 80대는 적정한가

0장에서 하남시 택시 80대의 평균 대기시간이 4.1분이라는 결과를 봤습니다.
이번에는 코드를 구성하는 클래스와 객체부터 확인한 뒤, 평균 뒤에 가려진 승객과 차량을 찾습니다.

교재 1.3절에 대응합니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, todo
from smartmob import Dtumos
from smartmob.viz import use_korean_font

use_korean_font()

## 1. 클래스에서 객체를 만듭니다

0장에서 `dt = Dtumos()` 를 실행했습니다. 이때 `Dtumos()` 가 무엇을 만들었는지 클래스 이름과 연결 상태로 확인합니다.

In [ ]:
dt = Dtumos()
print("dt의 클래스:", type(dt).__name__)
print("연결 상태:", dt.health())
print("사용 중인 모드:", dt.mode)

첫 줄에 `Dtumos` 가 출력됩니다. 클래스(class)는 같은 종류의 객체가 가질 데이터와 동작을 정의합니다. `Dtumos()` 로 만든 `dt` 는 객체(object)이며, 인스턴스(instance)라고도 부릅니다.

- `Dtumos` 는 클래스입니다.
- `dt = Dtumos()` 는 객체를 만들어 `dt` 에 담습니다.
- `dt.health()` 는 객체의 동작인 메서드(method)를 호출합니다.
- `dt.mode` 는 객체가 가진 속성(attribute)을 읽습니다.

메서드에는 `health()` 처럼 괄호가 붙고, 속성인 `mode` 에는 괄호가 없습니다.

## 2. 조건을 정하고 시뮬레이션을 실행합니다

시뮬레이션 조건을 딕셔너리 `REFERENCE` 에 모읍니다. `**REFERENCE` 는 딕셔너리에 담긴 조건을 `run_simulation()` 의 인자로 풀어 넣는 문법입니다.

In [ ]:
REFERENCE = dict(
    city="hanam", mode="taxi", fleet_size=80, num_passengers=1000,
    time_start=1080, time_end=1440, dispatch_mode="optimization",
    matrix_mode="street_distance", vehicle_capacity=1, random_seed=42,
)

sim = dt.run_simulation(**REFERENCE)
print("sim의 클래스:", type(sim).__name__)
sim.summary()

첫 줄에 `SimulationResult` 가 출력됩니다. `run_simulation()` 은 실행 결과를 이 클래스의 객체로 돌려줍니다. `summary()` 에는 호출 990건의 서비스율, 평균 대기시간, 차량 가동률이 들어 있습니다.

`sim.config` 에는 실행 조건, `sim.passengers` 에는 승객별 자료가 들어 있습니다. `sim.result` 에는 시각별 승객·차량 상태가 있고, `sim.summary()` 는 서비스율과 평균 대기시간 등을 계산합니다.

같은 `dt` 객체에 다른 조건을 넘기면 새 결과 객체를 만들 수 있습니다. 예를 들어 실서버를 사용할 때는 다음과 같이 차량 수만 바꿔 `sim` 과 `sim_40` 을 비교합니다. 녹화본에는 80대 결과만 있으므로 아래 코드는 설명용으로만 읽습니다.

```python
alternative = {**REFERENCE, "fleet_size": 40}
sim_40 = dt.run_simulation(**alternative)
```

## 3. 승객별 대기시간을 읽습니다

`sim.passengers` 는 승객마다 한 줄입니다. `status` 가 1이면 배차에 성공한 승객이고,
`wait_min` 은 호출부터 탑승까지 걸린 시간입니다.

In [ ]:
pax = sim.passengers
print("승객 수:", len(pax))
pax[["passenger_id", "status", "request_time", "pickup_time", "wait_min"]].head()

앞의 다섯 행에서 승객마다 호출 시각과 탑승 시각이 다름을 확인할 수 있습니다. 이제 배차에 성공한 990명의 `wait_min` 을 평균, 중앙값, 분위수로 요약합니다.

### 평균, 중앙값, 꼬리를 비교합니다

In [ ]:
waits = pax.loc[pax["status"] == 1, "wait_min"]

banner("대기시간 분포 (분)")
for label, value in [
    ("평균", waits.mean()),
    ("중앙값", waits.median()),
    ("상위 10%", waits.quantile(0.90)),
    ("상위 1%", waits.quantile(0.99)),
    ("최댓값", waits.max()),
]:
    print(f"{label:8s} {value:6.1f}")

평균과 중앙값은 붙어 있습니다. 그런데 최댓값은 평균의 몇 배입니다.
평균만 보고받은 사람은 이 승객의 존재를 모릅니다.

### 히스토그램으로 분포를 확인합니다

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(waits, bins=40, color="#4C6EF5", edgecolor="white")
ax.axvline(waits.mean(), color="crimson", linestyle="--", label=f"평균 {waits.mean():.1f}분")
ax.axvline(waits.quantile(0.90), color="black", linestyle=":", label=f"상위 10% {waits.quantile(0.90):.1f}분")
ax.set_xlabel("대기시간 (분)")
ax.set_ylabel("승객 수")
ax.legend()
ax.set_title("하남 택시 80대, 저녁 6시~자정")
plt.tight_layout();

왼쪽에 몰려 있고 오른쪽으로 길게 꼬리가 뻗습니다.
이런 모양에서는 평균만으로 모든 승객의 경험을 설명할 수 없습니다.

## 4. 오래 기다린 승객과 빈 차량을 찾습니다

먼저 꼬리에 있는 승객이 특정 시간대에 몰려 있는지 봅니다.

In [ ]:
from smartmob.data import minutes_to_hhmm

worst = pax.loc[pax["status"] == 1].nlargest(10, "wait_min")
worst = worst.assign(호출시각=worst["request_time"].map(minutes_to_hhmm))
worst[["passenger_id", "호출시각", "wait_min"]]

상위 10명의 호출은 18:38~19:39와 22:13~23:17에 나뉘어 있습니다. 특정 한 시각의 문제로 좁힐 수 없으므로, 이번에는 같은 결과 객체에서 차량 상태를 꺼내 함께 봅니다.

In [ ]:
vehicle_state = sim.result

banner("시간대별 평균 차량 수")
print(f"승객을 태운 차량  {vehicle_state['occupied_vehicle_num'].mean():5.1f}대")
print(f"승객에게 가는 차량 {vehicle_state['dispatched_vehicle_num'].mean():5.1f}대")
print(f"빈 차량          {vehicle_state['empty_vehicle_num'].mean():5.1f}대")

시간대별로 승객을 태운 차량은 평균 9.9대, 승객에게 가는 차량은 4.2대, 빈 차량은 31.3대입니다. 오래 기다린 승객이 있는데도 빈 차량이 많으므로 차량의 위치와 배차 규칙도 살펴봐야 합니다.

## 5. 직접 계산합니다

대기시간이 10분을 넘은 승객이 몇 명인지, 전체의 몇 퍼센트인지 구합니다.
그리고 그 승객들의 평균 대기시간도 구합니다.

In [ ]:
long_wait = None        # 10분을 넘게 기다린 승객 수 (명)
long_wait_share = None  # 전체 배차 승객 중 비율 (0~1)
long_wait_mean = None   # 그 승객들의 평균 대기시간 (분)

banner("빈칸 확인")
todo("10분 초과 승객", long_wait)
todo("그 비율", long_wait_share, fmt=lambda v: f"{v:.1%}")
todo("그들의 평균 대기", long_wait_mean, fmt=lambda v: f"{v:.1f}분")

## 6. 결과를 운영 판단으로 바꿉니다

80대 결과만으로 적정 차량 수를 확정할 수는 없습니다. 40대 결과와 비교하기 전에 어떤 지표를 볼지 아래 세 줄로 정리합니다. 답은 하나가 아닙니다.

1. 이 서비스를 "평균 4분"이라고 보고하면 무엇이 빠지는가
2. 대신 어떤 숫자를 함께 보고할 것인가. 그 이유는 무엇인가
3. 대기시간을 줄이려고 차량을 늘리면 무엇이 나빠지는가 (0장의 `utilization` 을 떠올립니다)

## 정리

- 클래스는 객체를 만드는 설계도이고, 객체의 동작은 메서드로 호출합니다
- `Dtumos` 객체에 실행 조건을 넘기면 `SimulationResult` 객체를 돌려받습니다
- 시뮬레이션 결과는 평균뿐 아니라 분포와 차량 상태를 함께 읽습니다
- `sim.passengers` 는 승객 단위, `sim.result` 는 시각 단위입니다
- 2장 실습에서는 이 시뮬레이션이 달리는 무대인 도로망 데이터를 직접 엽니다